# CatBoost 信贷风控建模

与逻辑回归保持相同的 **70% Train / 15% Selection / 15% Test** 切分逻辑。  
CatBoost 不复制 WOE、IV、VIF、P-value，而用原生类别变量、Early Stopping、复杂度搜索和 Train–Selection Gap 控制过拟合。  
**Test 只在模型参数锁定后使用；当前阶段只比较 AUC / AR / KS，不做 calibration。**

In [1]:
import numpy as np
import pandas as pd

from itertools import product
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

random_state = 42

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

# 如未安装：%pip install catboost -q


In [2]:
CSV_PATH = '/Users/ganlu/Desktop/home-credit-risk-english/data/application_train.csv'  # 改成你的本地路径

df = pd.read_csv(CSV_PATH)

assert 'TARGET' in df.columns
assert set(df['TARGET'].dropna().unique()).issubset({0, 1})

overview = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'bad_rate': df['TARGET'].mean(),
    'target_missing': df['TARGET'].isna().sum(),
    'duplicate_rows': df.duplicated().sum(),
    'unique_id': df['SK_ID_CURR'].nunique() if 'SK_ID_CURR' in df.columns else np.nan
})

overview


rows              307511.000000
columns              122.000000
bad_rate               0.080729
target_missing         0.000000
duplicate_rows         0.000000
unique_id         307511.000000
dtype: float64

In [3]:
# 与LR保持完全相同的样本切分
X = df.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
y = df['TARGET'].astype(int)

idx = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    idx,
    test_size=0.30,
    stratify=y,
    random_state=random_state
)

selection_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y.iloc[temp_idx],
    random_state=random_state
)

X_train = X.iloc[train_idx].copy()
X_selection = X.iloc[selection_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_selection = y.iloc[selection_idx].copy()
y_test = y.iloc[test_idx].copy()

pd.DataFrame({
    'sample': ['Train', 'Selection', 'Test'],
    'n': [len(y_train), len(y_selection), len(y_test)],
    'bad_rate': [y_train.mean(), y_selection.mean(), y_test.mean()]
})


,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test,46127,0.080734


In [ ]:
def scan_feature_quality(X, y):
    rows = []

    for col in X.columns:
        s = X[col]
        miss = s.isna()
        nunique = s.nunique(dropna=True)
        #normalize返回最高出现频率的数所占整个比例
        top_frequency = s.value_counts(
            dropna=False,
            normalize=True
        ).iloc[0]

        is_cat = (
            s.dtype == 'object'
            or str(s.dtype).startswith('string')
            or str(s.dtype).startswith('category')
        )

        if nunique < 2:
            flag = 'constant'
            drop = True

        elif top_frequency > 0.995:
            flag = 'quasi constant > 99.5%'
            drop = True

        elif is_cat and nunique / len(s) > 0.95:
            flag = 'ID-like categorical'
            drop = True

        elif miss.mean() > 0.95:
            flag = 'high missing > 95%'
            drop = False

        else:
            flag = ''
            drop = False

        rows.append({
            'feature': col,
            'dtype': str(s.dtype),
            'is_categorical': is_cat,
            'nunique': nunique,
            'missing_rate': miss.mean(),
            'top_frequency': top_frequency,
            'bad_rate_missing': y.loc[miss].mean() if miss.any() else np.nan,
            'bad_rate_nonmissing': y.loc[~miss].mean() if (~miss).any() else np.nan,
            'flag': flag,
            'drop': drop
        })

    return pd.DataFrame(rows)


feature_report = scan_feature_quality(
    X_train,
    y_train
)

basic_drop = feature_report.loc[
    feature_report['drop'],
    ['feature', 'flag']
].reset_index(drop=True)

final_features = feature_report.loc[
    ~feature_report['drop'],
    'feature'
].tolist()

print('原始特征数:', X_train.shape[1])
print('基础筛查剔除:', len(basic_drop))
print('进入CatBoost:', len(final_features))

basic_drop


原始特征数: 120
基础筛查剔除: 16
进入CatBoost: 104


,feature,flag
0,FLAG_MOBIL,quasi constant > 99.5%
1,FLAG_CONT_MOBILE,quasi constant > 99.5%
2,FLAG_DOCUMENT_2,quasi constant > 99.5%
3,FLAG_DOCUMENT_4,quasi constant > 99.5%
4,FLAG_DOCUMENT_7,quasi constant > 99.5%
5,FLAG_DOCUMENT_9,quasi constant > 99.5%
6,FLAG_DOCUMENT_10,quasi constant > 99.5%
7,FLAG_DOCUMENT_11,quasi constant > 99.5%
8,FLAG_DOCUMENT_12,quasi constant > 99.5%
9,FLAG_DOCUMENT_13,quasi constant > 99.5%


In [ ]:
X_train = X_train[final_features].copy()
X_selection = X_selection[final_features].copy()
X_test = X_test[final_features].copy()

cat_cols = X_train.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

# CatBoost原生类别变量，pool是catboost专用的容器
for data in (X_train, X_selection, X_test):
    for col in cat_cols:
        data[col] = (
            data[col]
            .astype('object')
            .fillna('__MISSING__')
            .astype(str)
        )

train_pool = Pool(
    X_train,
    y_train,
    cat_features=cat_cols
)

selection_pool = Pool(
    X_selection,
    y_selection,
    cat_features=cat_cols
)

test_pool = Pool(
    X_test,
    y_test,
    cat_features=cat_cols
)

print('类别变量数:', len(cat_cols))


类别变量数: 16


In [ ]:
#
def _psi_from_labels(a, b, eps=1e-6):
    #a，b每个箱（包含数值和类型分箱）
    a = pd.Series(a).astype(str).value_counts(normalize=True)
    b = pd.Series(b).astype(str).value_counts(normalize=True)
    #idx会统一分箱信息，a+b中所有的箱都会成为idx，clip为加上极小值保证不会报错
    idx = a.index.union(b.index)

    a = a.reindex(idx, fill_value=0).clip(lower=eps)
    b = b.reindex(idx, fill_value=0).clip(lower=eps)

    return float(
        ((a - b) * np.log(a / b)).sum()
    )


#Train vs Selection的分布形态，psi的返回结果应该相似。我们在进行train-test split的时候stratify=y只保证了
#坏样本的比例，但其他部分需要我们进一步验证
def feature_psi(train_s, other_s, bins=10):
    if pd.api.types.is_numeric_dtype(train_s):

        train_num = pd.to_numeric(
            train_s,
            errors='coerce'
        )

        other_num = pd.to_numeric(
            other_s,
            errors='coerce'
        )

        valid = train_num.dropna()

        if valid.nunique() >= 2:
            #创造10分位点，这部分和qcut有相似逻辑
            edges = np.unique(
                np.nanquantile(
                    valid,
                    np.linspace(0, 1, bins + 1)
                )
            ).astype(float)

            if len(edges) >= 3:
                edges[0] = -np.inf
                edges[-1] = np.inf

                a = pd.cut(
                    train_num,
                    edges,
                    include_lowest=True,
                    duplicates='drop'
                ).astype(object)

                b = pd.cut(
                    other_num,
                    edges,
                    include_lowest=True,
                    duplicates='drop'
                ).astype(object)

                a = pd.Series(
                    a,
                    index=train_s.index
                ).where(
                    train_num.notna(),
                    '__MISSING__'
                )

                b = pd.Series(
                    b,
                    index=other_s.index
                ).where(
                    other_num.notna(),
                    '__MISSING__'
                )

                return _psi_from_labels(a, b)

    return _psi_from_labels(
        pd.Series(train_s).fillna('__MISSING__'),
        pd.Series(other_s).fillna('__MISSING__')
    )


# 随机切分下PSI只做稳定性诊断，不用于删变量
psi_selection = pd.DataFrame({
    'feature': final_features,
    'psi_selection': [
        feature_psi(
            X_train[f],
            X_selection[f]
        )
        for f in final_features
    ]
}).sort_values(
    'psi_selection',
    ascending=False
).reset_index(drop=True)

psi_selection.head(20)


,feature,psi_selection
0,ORGANIZATION_TYPE,0.001516
1,OCCUPATION_TYPE,0.000698
2,DAYS_LAST_PHONE_CHANGE,0.000441
3,EXT_SOURCE_3,0.000397
4,COMMONAREA_AVG,0.000381
5,NAME_INCOME_TYPE,0.000377
6,NONLIVINGAREA_MEDI,0.000349
7,AMT_REQ_CREDIT_BUREAU_WEEK,0.000325
8,YEARS_BEGINEXPLUATATION_AVG,0.000323
9,YEARS_BEGINEXPLUATATION_MODE,0.000321


In [ ]:
def get_metrics(y, pred):
    auc = roc_auc_score(y, pred)
    fpr, tpr, _ = roc_curve(y, pred)

    return {
        'AUC': auc,
        'AR': 2 * auc - 1,
        'KS': np.max(tpr - fpr)
    }


def evaluate_model(
    model,
    X_train,
    y_train,
    X_selection,
    y_selection,
    X_test,
    y_test
):
    datasets = {
        'Train': (X_train, y_train),
        'Selection': (X_selection, y_selection),
        'Test': (X_test, y_test)
    }

    rows = []
#model = CatBoostClassifier(...)在进行fit前是我们写入的规则。而在model.fit后变成了一个固定好的函数，输入
#客户信息就会输出违约概率
#model.predict_proba(X_part)[:, 1]，对于X_part中的每一个客户预测属于每一个类别的概率，但是我们做的是二分类的任务
#所以只会返回两列，我们要拿出违约概率列，所以返回[:,1]
    for name, (X_part, y_part) in datasets.items():
        pred = model.predict_proba(X_part)[:, 1]
#**负责拆开传入的键值对
        rows.append({
            'sample': name,
            **get_metrics(y_part, pred)
        })

    return pd.DataFrame(rows).set_index('sample')


## Baseline

先跑一版固定参数，得到 CatBoost 的基准表现。

In [ ]:

#最好的方式是尽可能全在写在catboostclassifier里面，只留我训练的样本，和验证的样本在原来的fit里面（eval_set)
baseline_model = CatBoostClassifier(
    iterations=2500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_strength=1,
    bootstrap_type='Bernoulli',
    subsample=0.80,
    rsm=0.90,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=random_state,
    thread_count=-1,
    allow_writing_files=False,
    verbose=False
)

#catboostclassifier用于配置模型但还没有启动训练，后续会在fit中生效不用反复填写
baseline_model.fit(
    train_pool,
    #用哪份数据作为监控
    eval_set=selection_pool,
    #值保留验证集auc最好的轮次
    use_best_model=True,
    early_stopping_rounds=150
)

print(
    'Baseline best iteration:',
    baseline_model.get_best_iteration() + 1
)

baseline_train = get_metrics(
    y_train,
    baseline_model.predict_proba(X_train)[:, 1]
)

baseline_selection = get_metrics(
    y_selection,
    baseline_model.predict_proba(X_selection)[:, 1]
)

pd.DataFrame([{
    'depth': 6,
    'learning_rate': 0.03,
    'l2_leaf_reg': 5,
    'best_iteration': baseline_model.get_best_iteration() + 1,
    'train_ar': baseline_train['AR'],
    'selection_ar': baseline_selection['AR'],
    'ar_gap': baseline_train['AR'] - baseline_selection['AR'],
    'train_ks': baseline_train['KS'],
    'selection_ks': baseline_selection['KS']
}])


Baseline best iteration: 2132


,depth,learning_rate,l2_leaf_reg,best_iteration,train_ar,selection_ar,ar_gap,train_ks,selection_ks
0,6,0.03,5,2132,0.613836,0.521494,0.092342,0.460224,0.387015


In [ ]:
SEARCH_ITERATIONS = 3000
EARLY_STOPPING = 150
AR_TOLERANCE = 0.005


def run_candidate(params):
    model = CatBoostClassifier(
        iterations=SEARCH_ITERATIONS,
        #模型训练中的优化目标
        loss_function='Logloss',
        #early stopping的判断标志，如果不写则默认是loss function，但auc和loss并不完全划等号
        #auc是一个重要的是否出现过拟合的辅助判断信息
        eval_metric='AUC',
        #和subsample配套使用，每次采用该方法随机抽样，每个训练样本约80%参加
        bootstrap_type='Bernoulli',
        random_seed=random_state,
        thread_count=-1,
        allow_writing_files=False,
        verbose=False,
        **params
    )
    #use bestmodel会回填回最好的一次模型训练数据
    model.fit(
        train_pool,
        eval_set=selection_pool,
        use_best_model=True,
        early_stopping_rounds=EARLY_STOPPING
    )

    #
    train = get_metrics(
        y_train,
        #第二类返回的是模型预测为的概率
        model.predict_proba(X_train)[:, 1]
    )

    selection = get_metrics(
        y_selection,
        model.predict_proba(X_selection)[:, 1]
    )

    return {
        **params,
        'best_iteration': model.get_best_iteration() + 1,
        'train_ar': train['AR'],
        'selection_ar': selection['AR'],
        'ar_gap': abs(train['AR'] - selection['AR']),
        'train_ks': train['KS'],
        'selection_ks': selection['KS'],
        'ks_gap': abs(train['KS'] - selection['KS'])
    }


def run_search(param_list):
    rows = []

    for i, params in enumerate(param_list, start=1):
        print(f'{i}/{len(param_list)}', params)
        rows.append(run_candidate(params))

    return (
        pd.DataFrame(rows)
        .sort_values(
            ['selection_ar', 'ar_gap'],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )


def choose_best(
    result,
    ar_tolerance=AR_TOLERANCE
):
    best_ar = result['selection_ar'].max()

    # Selection AR接近最优时，优先选择泛化gap更小、结构更简单的模型
    #先确定ar最好的组合，往下ar差一些的容忍度内的组合也可以进来
    shortlist = result.loc[
        result['selection_ar']
        >= best_ar - ar_tolerance
    ].copy()

    return (
    #在入围的候选组合中，按照优先级进行排序
        shortlist
        .sort_values(
            [
                'ar_gap',
                'depth',
                'best_iteration',
                'selection_ar'
            ],
            ascending=[
                True,
                True,
                True,
                False
            ]
        )
        .iloc[0]
    )


## 第一轮：树深度

CatBoost 默认使用对称树，`depth` 会直接决定单棵树的复杂度。先固定其他参数，只寻找合适的深度区间。

In [ ]:
depth_candidates = [
    {
        'depth': depth,
        'learning_rate': 0.03,
        #叶节点的l2正则化
        'l2_leaf_reg': 5,
        #切点增加一点随机性，防止模型把极小噪声算进去
        'random_strength': 1,
        #构建树的时候只考虑80%的样本和90%的特征
        'subsample': 0.80,
        'rsm': 0.90
    }
    for depth in [4, 5, 6, 7, 8]
]

depth_result = run_search(
    depth_candidates
)

depth_best = choose_best(
    depth_result
)

display(depth_result)
depth_best


1/5 {'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
2/5 {'depth': 5, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
3/5 {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
4/5 {'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
5/5 {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}


,depth,learning_rate,l2_leaf_reg,random_strength,subsample,rsm,best_iteration,train_ar,selection_ar,ar_gap,train_ks,selection_ks,ks_gap
0,5,0.03,5,1,0.8,0.9,2816,0.599601,0.521820,0.077782,0.447518,0.385698,0.061820
1,6,0.03,5,1,0.8,0.9,2132,0.613836,0.521494,0.092342,0.460224,0.387015,0.073208
2,4,0.03,5,1,0.8,0.9,2346,0.562128,0.521073,0.041055,0.420038,0.385978,0.034060
3,7,0.03,5,1,0.8,0.9,2306,0.670597,0.521019,0.149579,0.504003,0.387865,0.116138
4,8,0.03,5,1,0.8,0.9,1383,0.664482,0.520402,0.144080,0.503177,0.387716,0.115461


depth                 4.000000
learning_rate         0.030000
l2_leaf_reg           5.000000
random_strength       1.000000
subsample             0.800000
rsm                   0.900000
best_iteration     2346.000000
train_ar              0.562128
selection_ar          0.521073
ar_gap                0.041055
train_ks              0.420038
selection_ks          0.385978
ks_gap                0.034060
Name: 2, dtype: float64

## 第二轮：Depth × Learning Rate × L2

第一轮只负责定位大致复杂度。第二轮不锁死单一 depth，而保留第一轮最佳 depth 及其相邻深度，再联合搜索 learning rate 和 L2，降低条件搜索错过更优组合的风险。

In [ ]:
anchor_depth = int(
    depth_best['depth']
)

tested_depths = [4, 5, 6, 7, 8]

fine_depths = [
    depth
    for depth in tested_depths
    if abs(depth - anchor_depth) <= 1
]
#人工拍定合理的学习率和正则化惩罚幅度
lr_l2_pairs = [
    (0.02, 3),
    (0.02, 5),
    (0.03, 3),
    (0.03, 5),
    (0.03, 10),
    (0.05, 5),
    (0.05, 10)
]

fine_candidates = [
    {
        'depth': depth,
        'learning_rate': lr,
        'l2_leaf_reg': l2,
        'random_strength': 1,
        'subsample': 0.80,
        'rsm': 0.90
    }
    for depth in fine_depths
    for lr, l2 in lr_l2_pairs
]

print('第二轮候选depth:', fine_depths)
print('第二轮模型数:', len(fine_candidates))

fine_result = run_search(
    fine_candidates
)

fine_best = choose_best(
    fine_result
)

display(fine_result)
fine_best


第二轮候选depth: [4, 5]
第二轮模型数: 14
1/14 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 3, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
2/14 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
3/14 {'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
4/14 {'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
5/14 {'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 10, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
6/14 {'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 5, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
7/14 {'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 10, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
8/14 {'depth': 5, 'learning_rate': 0.02, 'l2_leaf_reg': 3, 'random_strength': 1, 'subsample': 0.8, 'rsm': 0.9}
9/14 {'depth': 5, 'learning_rate': 0.02, 'l2_leaf_reg': 5, 'random_strength': 1,

,depth,learning_rate,l2_leaf_reg,random_strength,subsample,rsm,best_iteration,train_ar,selection_ar,ar_gap,train_ks,selection_ks,ks_gap
0,5,0.05,5,1,0.8,0.9,2022,0.612393,0.522077,0.090315,0.457709,0.388005,0.069705
1,5,0.03,10,1,0.8,0.9,2801,0.596619,0.521858,0.074761,0.446642,0.386498,0.060144
2,5,0.03,3,1,0.8,0.9,2463,0.591947,0.521838,0.070109,0.442508,0.388679,0.053829
3,5,0.03,5,1,0.8,0.9,2816,0.599601,0.521820,0.077782,0.447518,0.385698,0.061820
4,4,0.03,10,1,0.8,0.9,2816,0.569159,0.521430,0.047728,0.425093,0.385811,0.039282
5,5,0.02,3,1,0.8,0.9,2993,0.578535,0.521178,0.057357,0.430591,0.386240,0.044351
6,4,0.03,5,1,0.8,0.9,2346,0.562128,0.521073,0.041055,0.420038,0.385978,0.034060
7,4,0.03,3,1,0.8,0.9,2999,0.574679,0.520873,0.053807,0.429128,0.385157,0.043971
8,4,0.05,5,1,0.8,0.9,1563,0.567114,0.520794,0.046320,0.423033,0.386403,0.036630
9,5,0.02,5,1,0.8,0.9,2982,0.577436,0.520774,0.056661,0.432583,0.384362,0.048221


depth                 4.000000
learning_rate         0.020000
l2_leaf_reg           5.000000
random_strength       1.000000
subsample             0.800000
rsm                   0.900000
best_iteration     2964.000000
train_ar              0.555637
selection_ar          0.519508
ar_gap                0.036129
train_ks              0.415003
selection_ks          0.381738
ks_gap                0.033265
Name: 13, dtype: float64

## 第三轮：随机化与采样

在复杂度基本确定后，再微调样本采样、特征采样和随机扰动。

In [ ]:
#第三轮固定了fine best中的depth + learning rate + l2
base_params = {
    'depth':
    int(fine_best['depth']),

    'learning_rate':
    float(fine_best['learning_rate']),

    'l2_leaf_reg':
    float(fine_best['l2_leaf_reg'])
}

#这里只改动subsample，rsm和random strength（随机性强度）
#目标，通过增加随机性，减少单次模型的可用信息来防止过拟合，看在稳定核心模型后，如何能让模型范化能力更好
regularization_grid = [
    (0.70, 0.80, 0.5),
    (0.70, 0.90, 1.0),
    (0.80, 0.90, 1.0),
    (0.90, 0.90, 1.0),
    (0.80, 1.00, 0.5),
    (0.80, 0.80, 2.0)
]

#会把for中的内容套入字典作循环
regularization_candidates = [
    {
        **base_params,
        'subsample': subsample,
        'rsm': rsm,
        'random_strength': random_strength
    }
    for (
        subsample,
        rsm,
        random_strength
    ) in regularization_grid
]

regularization_result = run_search(
    regularization_candidates
)

all_search_result = pd.concat(
    [
        depth_result,
        fine_result,
        regularization_result
    ],
    ignore_index=True
).drop_duplicates(
    subset=[
        'depth',
        'learning_rate',
        'l2_leaf_reg',
        'random_strength',
        'subsample',
        'rsm'
    ]
).reset_index(drop=True)

#因为你不能保证第三轮的训练结果一定比第二轮第一轮好，所以应该从所有的结果中选择
best_row = choose_best(
    all_search_result
)

display(
    all_search_result
    .sort_values(
        'selection_ar',
        ascending=False
    )
    .head(30)
)

best_row


1/6 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5.0, 'subsample': 0.7, 'rsm': 0.8, 'random_strength': 0.5}
2/6 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5.0, 'subsample': 0.7, 'rsm': 0.9, 'random_strength': 1.0}
3/6 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5.0, 'subsample': 0.8, 'rsm': 0.9, 'random_strength': 1.0}
4/6 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5.0, 'subsample': 0.9, 'rsm': 0.9, 'random_strength': 1.0}
5/6 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5.0, 'subsample': 0.8, 'rsm': 1.0, 'random_strength': 0.5}
6/6 {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5.0, 'subsample': 0.8, 'rsm': 0.8, 'random_strength': 2.0}


,depth,learning_rate,l2_leaf_reg,random_strength,subsample,rsm,best_iteration,train_ar,selection_ar,ar_gap,train_ks,selection_ks,ks_gap
5,5,0.05,5.0,1.0,0.8,0.9,2022,0.612393,0.522077,0.090315,0.457709,0.388005,0.069705
6,5,0.03,10.0,1.0,0.8,0.9,2801,0.596619,0.521858,0.074761,0.446642,0.386498,0.060144
7,5,0.03,3.0,1.0,0.8,0.9,2463,0.591947,0.521838,0.070109,0.442508,0.388679,0.053829
0,5,0.03,5.0,1.0,0.8,0.9,2816,0.599601,0.521820,0.077782,0.447518,0.385698,0.061820
1,6,0.03,5.0,1.0,0.8,0.9,2132,0.613836,0.521494,0.092342,0.460224,0.387015,0.073208
8,4,0.03,10.0,1.0,0.8,0.9,2816,0.569159,0.521430,0.047728,0.425093,0.385811,0.039282
9,5,0.02,3.0,1.0,0.8,0.9,2993,0.578535,0.521178,0.057357,0.430591,0.386240,0.044351
2,4,0.03,5.0,1.0,0.8,0.9,2346,0.562128,0.521073,0.041055,0.420038,0.385978,0.034060
3,7,0.03,5.0,1.0,0.8,0.9,2306,0.670597,0.521019,0.149579,0.504003,0.387865,0.116138
10,4,0.03,3.0,1.0,0.8,0.9,2999,0.574679,0.520873,0.053807,0.429128,0.385157,0.043971


depth                 4.000000
learning_rate         0.020000
l2_leaf_reg           5.000000
random_strength       0.500000
subsample             0.700000
rsm                   0.800000
best_iteration     3000.000000
train_ar              0.554560
selection_ar          0.519057
ar_gap                0.035503
train_ks              0.415384
selection_ks          0.382869
ks_gap                0.032515
Name: 18, dtype: float64

## 锁定模型

Selection 已经完成参数选择和 Early Stopping。下面固定参数与树数，重新只用 Train 拟合一次；之后才打开 Test。

In [ ]:
#我们之前通过train和selection反复尝试调参以及最优迭代次数，现在按最终权重分类给一个最终模型
best_params = {
    'depth':
    int(best_row['depth']),

    'learning_rate':
    float(best_row['learning_rate']),

    'l2_leaf_reg':
    float(best_row['l2_leaf_reg']),

    'random_strength':
    float(best_row['random_strength']),

    'subsample':
    float(best_row['subsample']),

    'rsm':
    float(best_row['rsm'])
}

best_iterations = int(
    best_row['best_iteration']
)

final_model = CatBoostClassifier(
    iterations=best_iterations,
    bootstrap_type='Bernoulli',
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=random_state,
    thread_count=-1,
    allow_writing_files=False,
    verbose=100,
    **best_params
)

# 锁模后不再使用Selection做Early Stopping（通过观察selection集auc判断是否早停），跑完固定迭代轮次数
final_model.fit(
    train_pool
)

best_params, best_iterations


0:	total: 65.6ms	remaining: 3m 16s
100:	total: 6.26s	remaining: 2m 59s
200:	total: 12.7s	remaining: 2m 56s
300:	total: 19.1s	remaining: 2m 51s
400:	total: 25.6s	remaining: 2m 46s
500:	total: 32.5s	remaining: 2m 41s
600:	total: 39s	remaining: 2m 35s
700:	total: 45.6s	remaining: 2m 29s
800:	total: 52.2s	remaining: 2m 23s
900:	total: 58.9s	remaining: 2m 17s
1000:	total: 1m 5s	remaining: 2m 11s
1100:	total: 1m 12s	remaining: 2m 4s
1200:	total: 1m 18s	remaining: 1m 58s
1300:	total: 1m 25s	remaining: 1m 51s
1400:	total: 1m 32s	remaining: 1m 45s
1500:	total: 1m 38s	remaining: 1m 38s
1600:	total: 1m 45s	remaining: 1m 31s
1700:	total: 1m 51s	remaining: 1m 25s
1800:	total: 1m 58s	remaining: 1m 18s
1900:	total: 2m 5s	remaining: 1m 12s
2000:	total: 2m 11s	remaining: 1m 5s
2100:	total: 2m 18s	remaining: 59.1s
2200:	total: 2m 24s	remaining: 52.5s
2300:	total: 2m 31s	remaining: 45.9s
2400:	total: 2m 37s	remaining: 39.3s
2500:	total: 2m 44s	remaining: 32.7s
2600:	total: 2m 50s	remaining: 26.2s
2700:	t

({'depth': 4,
  'learning_rate': 0.02,
  'l2_leaf_reg': 5.0,
  'random_strength': 0.5,
  'subsample': 0.7,
  'rsm': 0.8},
 3000)

In [20]:
final_result = evaluate_model(
    final_model,
    X_train,
    y_train,
    X_selection,
    y_selection,
    X_test,
    y_test
)

final_result


,AUC,AR,KS
sample,,,
Train,0.777280,0.554560,0.415384
Selection,0.759528,0.519057,0.382869
Test,0.763668,0.527336,0.395380


In [ ]:
# 锁模后再看Test稳定性
#这里psi看的分布是训练集和测试集中的分布
psi_test = pd.DataFrame({
    'feature': final_features,
    'psi_test': [
        feature_psi(
            X_train[f],
            X_test[f]
        )
        for f in final_features
    ]
})

psi_table = (
    psi_selection
    .merge(
        psi_test,
        on='feature',
        how='left'
    )
)

psi_table['max_psi'] = psi_table[
    ['psi_selection', 'psi_test']
].max(axis=1)

psi_table['psi_flag'] = np.select(
    [
        psi_table['max_psi'] >= 0.25,
        psi_table['max_psi'] >= 0.10
    ],
    [
        'shift',
        'watch'
    ],
    default='stable'
)

psi_table.sort_values(
    'max_psi',
    ascending=False
).head(30)


,feature,psi_selection,psi_test,max_psi,psi_flag
0,ORGANIZATION_TYPE,0.001516,0.001486,0.001516,stable
1,OCCUPATION_TYPE,0.000698,0.000335,0.000698,stable
35,LIVINGAREA_MEDI,0.000174,0.000482,0.000482,stable
59,DAYS_ID_PUBLISH,0.000112,0.000467,0.000467,stable
56,LIVINGAREA_AVG,0.000119,0.000460,0.000460,stable
2,DAYS_LAST_PHONE_CHANGE,0.000441,0.000325,0.000441,stable
3,EXT_SOURCE_3,0.000397,0.000384,0.000397,stable
4,COMMONAREA_AVG,0.000381,0.000148,0.000381,stable
5,NAME_INCOME_TYPE,0.000377,0.000291,0.000377,stable
49,AMT_INCOME_TOTAL,0.000143,0.000360,0.000360,stable


In [17]:
feature_importance = pd.DataFrame({
    'feature':
    final_model.feature_names_,

    'importance':
    final_model.get_feature_importance()
}).sort_values(
    'importance',
    ascending=False
).reset_index(drop=True)

feature_importance.head(30)


,feature,importance
0,EXT_SOURCE_3,24.112386
1,EXT_SOURCE_2,18.338312
2,EXT_SOURCE_1,7.896534
3,DAYS_BIRTH,5.229409
4,AMT_GOODS_PRICE,4.726411
5,AMT_CREDIT,4.321407
6,AMT_ANNUITY,3.234260
7,DAYS_EMPLOYED,2.916771
8,CODE_GENDER,2.488124
9,NAME_EDUCATION_TYPE,2.222927


In [ ]:
# 锁模后用Test做最终解释
shap_n = min(
    3000,
    len(X_test)
)

shap_idx = X_test.sample(
    n=shap_n,
    random_state=random_state
).index

shap_pool = Pool(
    X_test.loc[shap_idx],
    y_test.loc[shap_idx],
    cat_features=cat_cols
)

#即使什么都不写get_feature_importance里默认type = shapvalues
shap_values = (
    final_model
    .get_feature_importance(
        shap_pool,
        type='ShapValues'
    )[:, :-1]
)

shap_importance = pd.DataFrame({
    'feature':
    final_model.feature_names_,

    'mean_abs_shap':
    np.abs(shap_values).mean(axis=0)
}).sort_values(
    'mean_abs_shap',
    ascending=False
).reset_index(drop=True)

shap_importance.head(30)


,feature,mean_abs_shap
0,EXT_SOURCE_3,0.370289
1,EXT_SOURCE_2,0.319825
2,AMT_GOODS_PRICE,0.199628
3,EXT_SOURCE_1,0.170952
4,AMT_CREDIT,0.147028
5,DAYS_EMPLOYED,0.126990
6,CODE_GENDER,0.118548
7,NAME_EDUCATION_TYPE,0.097112
8,AMT_ANNUITY,0.095818
9,DAYS_BIRTH,0.093101


In [19]:
model_summary = {
    'train_n':
    len(X_train),

    'selection_n':
    len(X_selection),

    'test_n':
    len(X_test),

    'final_features':
    len(final_features),

    'categorical_features':
    len(cat_cols),

    'best_iterations':
    best_iterations,

    **best_params
}

model_summary

# final_model.save_model('catboost_credit_risk.cbm')


{'train_n': 215257,
 'selection_n': 46127,
 'test_n': 46127,
 'final_features': 104,
 'categorical_features': 16,
 'best_iterations': 3000,
 'depth': 4,
 'learning_rate': 0.02,
 'l2_leaf_reg': 5.0,
 'random_strength': 0.5,
 'subsample': 0.7,
 'rsm': 0.8}